# Qwen3-4B Japanese → English Light Novel Translation Fine-Tuning
Dual T4 Kaggle notebook

This notebook trains Qwen3-4B as a domain-specialized translator using:
- pre-tokenized Arrow datasets produced locally
- QLoRA (4-bit base + LoRA adapters)
- DDP across 2 GPUs via accelerate.notebook_launcher
- gradient checkpointing
- packed causal LM training
- checkpoint resume support
- custom throughput / GPU memory logging


In [ ]:
!pip install -q \
    transformers==4.57.1 \
    accelerate==1.10.1 \
    datasets==4.0.0 \
    peft==0.17.1 \
    trl==0.21.0 \
    bitsandbytes==0.47.0 \
    sentencepiece \
    safetensors

In [1]:
import os
import gc
import math
import time
import json
import glob
import random
import subprocess
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import torch
from torch.utils.data import Dataset as TorchDataset

from datasets import load_from_disk, concatenate_datasets
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    TrainerCallback,
)
from peft import (
    LoraConfig,
    TaskType,
    get_peft_model,
    prepare_model_for_kbit_training,
    PeftModel,
)
from transformers.trainer_utils import get_last_checkpoint

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["WANDB_DISABLED"] = "true"

torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.benchmark = True

# Accelerate will handle the distributed worker launch in the standalone script.


2026-06-05 00:05:48.883664: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1780617949.304775     128 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1780617949.452888     128 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1780617950.400521     128 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780617950.400554     128 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780617950.400557     128 computation_placer.cc:177] computation placer alr

In [3]:
# The training is launched from a standalone script via Accelerate.
# Keeping this notebook process single-process avoids fork/spawn issues inside Kaggle notebooks.
print("Accelerate launch is handled by the external training script.")

multiprocessing start method: spawn


In [ ]:
!wget https://3be9-2406-7400-35-9b98-4f4a-4f43-1a21-1ba6.ngrok-free.app/PreProcessingOutputs.zip -o /kaggle/working/Dataset.zip -v

In [ ]:
!unzip /kaggle/working/PreProcessingOutputs.zip -d /kaggle/working/tokenized_dataset

In [ ]:
!mv /kaggle/working/tokenized_dataset/PreProcessingOutputs/Arifureta-Novel/* /kaggle/working/tokenized_dataset_1

In [4]:
BASE_MODEL = "Qwen/Qwen3-4B"
# Point this to the directory produced by the preprocessing script.
TOKENIZED_ROOT = Path("/kaggle/working/tokenized_dataset_1")  # change for your dataset mount
OUTPUT_ROOT = Path("/kaggle/working/qwen3_ln_translation")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# Training lengths
MAX_SEQ_LEN = 3072  # good compromise for dual T4 + long narrative context
FULL_WEIGHT = 0.2   # curriculum phase 1
CHUNK_WEIGHT = 0.8
PHASES = [
    {"name": "phase1_chunk_heavy", "full_w": 0.2, "chunk_w": 0.8, "epochs": 1},
    {"name": "phase2_balanced",    "full_w": 0.5, "chunk_w": 0.5, "epochs": 1},
    {"name": "phase3_full_heavy",  "full_w": 0.8, "chunk_w": 0.2, "epochs": 1},
]

# Optimization
PER_DEVICE_TRAIN_BATCH_SIZE = 2
PER_DEVICE_EVAL_BATCH_SIZE = 2
GRAD_ACCUM_STEPS = 8
LR = 1.5e-4
WARMUP_RATIO = 0.03
WEIGHT_DECAY = 0.0
MAX_GRAD_NORM = 1.0
LOGGING_STEPS = 20
SAVE_STEPS = 1000
EVAL_STEPS = 1000
SEED = 42

# LoRA
LORA_R = 32
LORA_ALPHA = 64
LORA_DROPOUT = 0.05

# 4-bit quantization for QLoRA
USE_4BIT = True


In [5]:
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


In [6]:
def gpu_stats() -> List[Dict]:
    stats = []
    try:
        import pynvml
        pynvml.nvmlInit()
        n = pynvml.nvmlDeviceGetCount()
        for i in range(n):
            h = pynvml.nvmlDeviceGetHandleByIndex(i)
            mem = pynvml.nvmlDeviceGetMemoryInfo(h)
            util = pynvml.nvmlDeviceGetUtilizationRates(h)
            stats.append({
                "gpu": i,
                "utilization_percent": int(util.gpu),
                "memory_used_mb": round(mem.used / (1024**2), 1),
                "memory_total_mb": round(mem.total / (1024**2), 1),
            })
    except Exception:
        # Fallback to nvidia-smi.
        try:
            out = subprocess.check_output(
                ["bash", "-lc", "nvidia-smi --query-gpu=index,utilization.gpu,memory.used,memory.total --format=csv,noheader,nounits"],
                text=True,
            ).strip().splitlines()
            for line in out:
                if not line.strip():
                    continue
                idx, util, used, total = [x.strip() for x in line.split(",")]
                stats.append({
                    "gpu": int(idx),
                    "utilization_percent": int(util),
                    "memory_used_mb": float(used),
                    "memory_total_mb": float(total),
                })
        except Exception:
            pass
    return stats


In [7]:
def load_split(path: Path):
    return load_from_disk(str(path))

full_train = load_split(TOKENIZED_ROOT / "full_chapter" / "train")
full_val   = load_split(TOKENIZED_ROOT / "full_chapter" / "validation")
chunk_train = load_split(TOKENIZED_ROOT / "chunked" / "train")
chunk_val   = load_split(TOKENIZED_ROOT / "chunked" / "validation")

print("Loaded:")
print("full_train:", len(full_train), "full_val:", len(full_val))
print("chunk_train:", len(chunk_train), "chunk_val:", len(chunk_val))


Loaded:
full_train: 4275 full_val: 221
chunk_train: 5483 chunk_val: 280


In [8]:
def sample_dataset(ds, n: int, seed: int):
    """Sample n rows, with replacement when needed."""
    if n <= 0 or len(ds) == 0:
        return ds.select([])
    rng = np.random.default_rng(seed)
    if n <= len(ds):
        idx = rng.permutation(len(ds))[:n].tolist()
    else:
        idx = rng.integers(0, len(ds), size=n).tolist()
    return ds.select(idx)

def build_mixture(full_ds, chunk_ds, full_w: float, chunk_w: float, seed: int):
    """
    Build a weighted mixture for a curriculum phase.
    We oversample the smaller side when needed so the ratio is respected.
    """
    if len(full_ds) == 0 and len(chunk_ds) == 0:
        raise ValueError("Both datasets are empty.")

    total_target = max(
        math.ceil(len(full_ds) / max(full_w, 1e-9)) if len(full_ds) else 0,
        math.ceil(len(chunk_ds) / max(chunk_w, 1e-9)) if len(chunk_ds) else 0,
    )
    n_full = max(0, int(round(total_target * full_w)))
    n_chunk = max(0, int(round(total_target * chunk_w)))

    # Ensure we never end up with an empty curriculum when a side exists.
    if len(full_ds) and n_full == 0:
        n_full = 1
    if len(chunk_ds) and n_chunk == 0:
        n_chunk = 1

    full_sel = sample_dataset(full_ds, n_full, seed=seed + 1)
    chunk_sel = sample_dataset(chunk_ds, n_chunk, seed=seed + 2)

    parts = []
    if len(full_sel):
        parts.append(full_sel)
    if len(chunk_sel):
        parts.append(chunk_sel)

    if len(parts) == 1:
        mixed = parts[0]
    else:
        mixed = concatenate_datasets(parts)

    mixed = mixed.shuffle(seed=seed)
    return mixed


In [9]:
def choose_attention_impl() -> str:
    """
    FlashAttention2 is great when available, but T4 often benefits from falling back to SDPA.
    The notebook auto-selects the best supported attention implementation.
    """
    try:
        import importlib.util
        if importlib.util.find_spec("flash_attn") is not None:
            major, minor = torch.cuda.get_device_capability()
            # Conservative gate: only enable FA2 on Ampere+.
            if major >= 8:
                return "flash_attention_2"
    except Exception:
        pass
    return "sdpa"



def load_tokenizer(model_name: str):
    tok = AutoTokenizer.from_pretrained(model_name, use_fast=True)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    tok.padding_side = "right"
    return tok

def load_model_for_training(model_name: str, local_rank: int):
    """
    Load Qwen3-4B in 4-bit if enabled, then attach LoRA adapters.
    Each DDP worker loads onto its own GPU only.
    """
    quant_config = None
    if USE_4BIT:
        quant_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.float16,
        )

    device_map = {"": local_rank} if torch.cuda.is_available() else None

    # Important: this function must run only inside spawned worker processes.
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16,
        quantization_config=quant_config,
        device_map=device_map,
        attn_implementation=ATTN_IMPL,
        low_cpu_mem_usage=True,
    )
    model.config.use_cache = False
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

    # Qwen-family linear projections are generally named like this.
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ]

    lora_cfg = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        bias="none",
        target_modules=target_modules,
        inference_mode=False,
    )
    model = get_peft_model(model, lora_cfg)
    model.print_trainable_parameters()
    return model


In [10]:
class PackingCausalCollator:
    """
    Packs already-tokenized examples into fixed-length sequences without any re-tokenization.
    Each example must already contain input_ids, attention_mask, and labels.
    """
    def __init__(self, tokenizer, max_length: int):
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id
        if self.pad_id is None:
            self.pad_id = 0

    def __call__(self, features: List[Dict]) -> Dict[str, torch.Tensor]:
        blocks_input_ids = []
        blocks_attention = []
        blocks_labels = []

        cur_ids: List[int] = []
        cur_labels: List[int] = []

        def flush():
            if not cur_ids:
                return
            pad_len = self.max_length - len(cur_ids)
            blocks_input_ids.append(cur_ids + [self.pad_id] * pad_len)
            blocks_attention.append([1] * len(cur_ids) + [0] * pad_len)
            blocks_labels.append(cur_labels + [-100] * pad_len)

        for feat in features:
            ids = feat["input_ids"]
            labels = feat["labels"]

            # Ensure Python lists.
            if isinstance(ids, torch.Tensor):
                ids = ids.tolist()
            if isinstance(labels, torch.Tensor):
                labels = labels.tolist()

            # Extremely long examples are split at arbitrary boundaries only as a safety fallback.
            start = 0
            while start < len(ids):
                remaining = self.max_length - len(cur_ids)
                take = min(remaining, len(ids) - start)
                cur_ids.extend(ids[start:start + take])
                cur_labels.extend(labels[start:start + take])
                start += take

                if len(cur_ids) == self.max_length:
                    flush()
                    cur_ids = []
                    cur_labels = []

        flush()

        if not blocks_input_ids:
            # Defensive fallback.
            blocks_input_ids = [[self.pad_id] * self.max_length]
            blocks_attention = [[0] * self.max_length]
            blocks_labels = [[-100] * self.max_length]

        return {
            "input_ids": torch.tensor(blocks_input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(blocks_attention, dtype=torch.long),
            "labels": torch.tensor(blocks_labels, dtype=torch.long),
        }


In [11]:
class TokenThroughputTrainer(Trainer):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self._train_start_time = None
        self._tokens_seen = 0

    def training_step(self, model, inputs, num_items_in_batch=None):
        labels = inputs.get("labels")
        if labels is not None:
            with torch.no_grad():
                self._tokens_seen += int((labels != -100).sum().item())
        if self._train_start_time is None:
            self._train_start_time = time.time()
        return super().training_step(model, inputs, num_items_in_batch=num_items_in_batch)

    def log(self, logs: Dict[str, float], start_time: Optional[float] = None):
        logs = dict(logs)
        if self._train_start_time is not None:
            elapsed = max(1e-6, time.time() - self._train_start_time)
            logs["train_tokens_seen"] = self._tokens_seen
            logs["tokens_per_sec"] = self._tokens_seen / elapsed
        logs["gpu_stats"] = json.dumps(gpu_stats(), ensure_ascii=False)
        return super().log(logs, start_time=start_time)


In [12]:
def train_phase(
    phase_cfg: Dict,
    seed: int,
    init_adapter: Optional[str] = None,
    resume_from_checkpoint: Optional[str] = None,
):
    set_seed(seed)

    phase_name = phase_cfg["name"]
    phase_dir = OUTPUT_ROOT / phase_name
    phase_dir.mkdir(parents=True, exist_ok=True)

    tokenizer = load_tokenizer(BASE_MODEL)

    train_ds = build_mixture(full_train, chunk_train, phase_cfg["full_w"], phase_cfg["chunk_w"], seed=seed)
    eval_ds  = build_mixture(full_val, chunk_val, phase_cfg["full_w"], phase_cfg["chunk_w"], seed=seed + 1000)

    print(f"[{phase_name}] train samples={len(train_ds)} eval samples={len(eval_ds)}")

    local_rank = int(os.environ.get("LOCAL_RANK", "0"))
    base_model = load_model_for_training(BASE_MODEL, local_rank=local_rank)

    # Continue training from the previous phase's adapter when available.
    if init_adapter is not None:
        model = PeftModel.from_pretrained(base_model, init_adapter, is_trainable=True)
    else:
        model = base_model

    collator = PackingCausalCollator(tokenizer, max_length=MAX_SEQ_LEN)

    args = TrainingArguments(
        output_dir=str(phase_dir),
        num_train_epochs=phase_cfg["epochs"],
        per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM_STEPS,
        learning_rate=LR,
        warmup_ratio=WARMUP_RATIO,
        weight_decay=WEIGHT_DECAY,
        lr_scheduler_type="cosine",
        logging_steps=LOGGING_STEPS,
        save_steps=SAVE_STEPS,
        eval_steps=EVAL_STEPS,
        save_strategy="steps",
        evaluation_strategy="steps",
        save_total_limit=2,
        fp16=True,
        bf16=False,
        optim="paged_adamw_8bit",
        max_grad_norm=MAX_GRAD_NORM,
        report_to=["tensorboard"],
        logging_dir=str(phase_dir / "logs"),
        seed=seed,
        data_seed=seed,
        remove_unused_columns=False,
        dataloader_num_workers=4,
        dataloader_pin_memory=True,
        ddp_find_unused_parameters=False,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        group_by_length=False,
        label_names=["labels"],
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        save_safetensors=True,
    )

    trainer = TokenThroughputTrainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=eval_ds,
        data_collator=collator,
        tokenizer=tokenizer,
    )

    # Resume from the last checkpoint if provided or if one already exists.
    if resume_from_checkpoint is None:
        last = get_last_checkpoint(str(phase_dir))
        resume_from_checkpoint = last

    train_result = trainer.train(resume_from_checkpoint=resume_from_checkpoint)
    metrics = train_result.metrics
    metrics["train_samples"] = len(train_ds)
    metrics["eval_samples"] = len(eval_ds)

    final_adapter_dir = phase_dir / "final_adapter"
    trainer.save_model(str(final_adapter_dir))
    tokenizer.save_pretrained(str(final_adapter_dir))

    with open(phase_dir / "metrics.json", "w", encoding="utf-8") as f:
        json.dump(metrics, f, ensure_ascii=False, indent=2)

    eval_metrics = trainer.evaluate()
    with open(phase_dir / "eval_metrics.json", "w", encoding="utf-8") as f:
        json.dump(eval_metrics, f, ensure_ascii=False, indent=2)

    del trainer, model, base_model
    gc.collect()
    torch.cuda.empty_cache()

    return str(final_adapter_dir)

def train_worker(process_index: int = 0):
    global ATTN_IMPL

    local_rank = int(os.environ.get("LOCAL_RANK", process_index))
    if torch.cuda.is_available():
        torch.cuda.set_device(local_rank)

    ATTN_IMPL = choose_attention_impl()

    print(
        f"[rank {local_rank}] "
        f"attention_impl={ATTN_IMPL}"
    )
    set_seed(SEED + process_index)
    last_adapter = None
    for idx, phase in enumerate(PHASES):
        phase_seed = SEED + idx * 100 + process_index
        last_adapter = train_phase(
            phase,
            seed=phase_seed,
            init_adapter=last_adapter,
            resume_from_checkpoint=None,
        )
    return last_adapter


In [13]:
# Launch the standalone LoRA/QLoRA training script with Accelerate.
# This is the only process-spawning step; the script itself uses adapter-only fine-tuning safely under Accelerate.
!accelerate launch --num_processes 2 --multi_gpu --mixed_precision fp16 /kaggle/working/train_qwen3_ln_translation_accelerate_lora.py

Launching training on 2 CUDAs.
[rank 0] attention_impl=sdpa
[rank 1] attention_impl=sdpa


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

...

model-00001-of-00003.safetensors:   0%|          | 0.00/3.96G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/99.6M [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/3.99G [00:00<?, ?B/s]

W0605 00:07:46.333000 128 torch/multiprocessing/spawn.py:165] Terminating process 192 via signal SIGTERM
E0605 00:07:46.391000 128 torch/distributed/elastic/multiprocessing/api.py:833] failed (exitcode: 1) local_rank: 0 (pid: 189) of fn: train_worker (start_method: fork)
E0605 00:07:46.391000 128 torch/distributed/elastic/multiprocessing/api.py:833] Traceback (most recent call last):
E0605 00:07:46.391000 128 torch/distributed/elastic/multiprocessing/api.py:833]   File "/usr/local/lib/python3.12/dist-packages/torch/distributed/elastic/multiprocessing/api.py", line 788, in _poll
E0605 00:07:46.391000 128 torch/distributed/elastic/multiprocessing/api.py:833]     self._pc.join(-1)
E0605 00:07:46.391000 128 torch/distributed/elastic/multiprocessing/api.py:833]   File "/usr/local/lib/python3.12/dist-packages/torch/multiprocessing/spawn.py", line 211, in join
E0605 00:07:46.391000 128 torch/distributed/elastic/multiprocessing/api.py:833]     raise ProcessRaisedException(msg, error_index, fai

ChildFailedError: 
============================================================
train_worker FAILED
------------------------------------------------------------
Failures:
  <NO_OTHER_FAILURES>
------------------------------------------------------------
Root Cause (first observed failure):
[0]:
  time      : 2026-06-05_00:07:46
  host      : 9b7196e2e3bf
  rank      : 0 (local_rank: 0)
  exitcode  : 1 (pid: 189)
  error_file: /tmp/torchelastic_5ptuqvln/none_0mrwwaqm/attempt_0/0/error.json
  traceback : Traceback (most recent call last):
    File "/usr/local/lib/python3.12/dist-packages/torch/distributed/elastic/multiprocessing/errors/__init__.py", line 362, in wrapper
      return f(*args, **kwargs)
             ^^^^^^^^^^^^^^^^^^
    File "/tmp/ipykernel_128/2960103105.py", line 118, in train_worker
      last_adapter = train_phase(
                     ^^^^^^^^^^^^
    File "/tmp/ipykernel_128/2960103105.py", line 21, in train_phase
      base_model = load_model_for_training(BASE_MOD

In [ ]:
# Merge is now done inside /kaggle/working/train_qwen3_ln_translation_accelerate_lora.py
# on rank 0 after the final phase completes.

In [ ]:
prompt = """Translate the following Japanese light novel passage into natural English light novel prose.

Preserve meaning, tone, dialogue, names, honorific intent, and paragraph breaks.
Output only the English translation.

Japanese:
今日は本当に長い一日だった。

English:
"""
inputs = tokenizer(prompt, return_tensors="pt").to(merged_model.device)
with torch.no_grad():
    out = merged_model.generate(
        **inputs,
        max_new_tokens=128,
        do_sample=False,
        temperature=0.0,
        pad_token_id=tokenizer.eos_token_id,
    )
print(tokenizer.decode(out[0], skip_special_tokens=True))
